crear un modelo de agente que resuelva el siguiente problema, tengo una recuadro de 10 x 6 y tengo de 10 fichas de cuadrados 2x2, L 3x2, s 3x2 y barra 4x1, tengo que ver la manera de que mi agente acomode esas piesas sin dejar ningun espacio

In [52]:
import numpy as np

In [53]:
ALTURA = 6
ANCHO  = 10 

In [54]:
#Ponemos puntos en un eje cartesiano para armanr
cuadrado = [(0, 0), (1, 0),
            (0, 1), (1, 1)]

L = [(0, 0), (1, 0), (2, 0),
     (0, 1)]

S = [(1, 0), (2, 0),
             (0, 1), (1, 1)]

barra = [(0, 0), (1, 0), (2, 0), (3, 0)]



In [55]:
def normalizar(shape):
    min_x = min(x for x, y in shape)
    min_y = min(y for x, y in shape)
    return sorted([(x - min_x, y - min_y) for x, y in shape])


esto es para rotar si es que no entra en alguna de las casillas

In [56]:
def rotar_90(shape):
    rotada = [(y, -x) for x, y in shape]
    return normalizar(rotada)

Genera hasta 4 orientaciones (rotaciones) distintas de una pieza.

In [57]:
def mover(shape_base):
    formas = []
    actual = normalizar(shape_base)
    vistos = set()

    for _ in range(4):
        tupla = tuple(actual)
        if tupla not in vistos:
            vistos.add(tupla)
            formas.append(actual)
        actual = rotar_90(actual)

    return formas


Orden en el que el agente probará las piezas
POLIITICA

In [58]:
PIEZAS = {
    "Q": mover(cuadrado),
    "L": mover(L),
    "S": mover(S),
    "I": mover(barra)
}
ORDEN_PIEZAS = ["Q", "L", "S", "I"]

Definimos nuestro entorno

In [ ]:
def crear_tablero():
    return np.zeros((ALTURA, ANCHO), dtype=int)


def mostrar_tablero(tablero):
    chars = {0: "."}
    for i, nombre in enumerate(ORDEN_PIEZAS, start=1):
        chars[i] = nombre

    for fila in range(ALTURA):
        linea = ""
        for col in range(ANCHO):
            linea += chars[tablero[fila, col]] + " "
        print(linea)
    print()

def primera_celda_vacia(tablero):
    """Busca la primera celda vacía (política de selección de celda)."""
    for y in range(ALTURA):
        for x in range(ANCHO):
            if tablero[y, x] == 0:
                return x, y
    return None  # tablero lleno


def se_puede_colocar(tablero, forma, ox, oy):
    for dx, dy in forma:
        x = ox + dx
        y = oy + dy
        # Fuera del tablero
        if x < 0 or x >= ANCHO or y < 0 or y >= ALTURA:
            return False
        # Celda ocupada
        if tablero[y, x] != 0:
            return False
    return True

def colocar(tablero, forma, ox, oy, id_pieza):
    for dx, dy in forma:
        x = ox + dx
        y = oy + dy
        tablero[y, x] = id_pieza


def quitar(tablero, forma, ox, oy):
    for dx, dy in forma:
        x = ox + dx
        y = oy + dy
        tablero[y, x] = 0



In [ ]:
def agente_resuelve(tablero):
    #Vemos si ya está llena las celdas
    celda = primera_celda_vacia(tablero)
    if celda is None:
        return True  # sin huecos -> éxito

    x0, y0 = celda

    #El agente prueba pezas segun la politica ya dicha
    for idx_nombre, nombre in enumerate(ORDEN_PIEZAS, start=1):
        formas = PIEZAS[nombre]

        #Para cada orientación, intenta colocar la pieza con origen en (x0, y0)
        for forma in formas:
            if se_puede_colocar(tablero, forma, x0, y0):
                # Colocamos la pieza con id = idx_nombre
                colocar(tablero, forma, x0, y0, idx_nombre)

                #seguir llenando
                if agente_resuelve(tablero):
                    return True

            

    # Si ninguna pieza encaja en esta celda, no hay solución por este camino
    return False

